Everytime you want to check the high energy (HE) scale with Thorium data, you first need to create two files:

- 3D map with $^{83\text{m}}\text{Kr}$ data previous to HE calibration campaign.
- H5 file with energy scale time evolution within the HE calibration campaign.

__NOTE:__ _this is valid until we get more details about Zemrude city._

In [1]:
import sys
sys.path.append('/lhome/ific/c/ccortesp/Analysis')

from libs import crudo

import glob
import numpy as np
import os
import pandas as pd

# Styling Plot
crudo.pt.ccortesp_plot_style()

%matplotlib inline
%load_ext autoreload
%autoreload 2

Crudo package loaded successfully.
Available sub-modules: data_management (dm), energy_functions (ef), fit_functions (ff), plotting_tools (pt), topology_functions (tf), utilities (ut).


# Preliminary

In [15]:
# --- DIRECTORIES
ICAROS_LOW_BCKG_DIR = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Icaros/Low_background/2025/'
ICAROS_HE_DIR = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Icaros/HE_calibration/2025/'
OUTPUT_DIR = '/lhome/ific/c/ccortesp/Analysis/NEXT-100/Th_analysis/h5/'

# --- FILES
RUNS_TO_COMBINE = [15546, 15547, 15557]      # Low-background runs to combine their maps
RUNS_FOR_ENERGY_SCALE = [15589]              # HE calibration runs to use for energy scale time evolution

# --- FUNCTIONS
def merge_bins(df):
    """
    Merges data for a single bin (voxel) using a weighted average.
    """
    new_n  = df.nevents.sum()
    new_mu = np.sum(df.mu * df.nevents) / np.clip(new_n, 1, None)
    return pd.DataFrame(dict(mu=new_mu, nevents=new_n), index=[0])
                       

def combine_maps(maps_list):
    """
    Combines a list of Kr map DataFrames into a single, high-statistics map.
    """
    full_map = pd.concat(maps_list, ignore_index=True)    
    combined_map = full_map.groupby(["dt", "x", "y"]).apply(merge_bins).reset_index()
    return combined_map

def create_full_h5_scale(scale_list):
    """
    Combines a list of Kr time evolution DataFrames into a single DataFrame.
    It contains the time evolution of the energy scale (or yield) S2e.
    """
    full_scale = pd.concat(scale_list, ignore_index=True)
    combined_scale = full_scale[['run_number', 'ts', 's2e', 's2eu']].rename(columns={'run_number': 'run', 'ts': 'time'})
    return combined_scale

### Testing

In [ ]:
test_file = '/lhome/ific/c/ccortesp/Analysis/NEXT-100/Th_analysis/h5/energy_scale_he.h5'
# test_file = '/lhome/ific/c/ccortesp/Analysis/NEXT-100/Th_analysis/h5/combined_15546_15557.map3d'

test_df = pd.read_hdf(test_file, key='data')
test_df

# Combine 3D Map

In [14]:
# List of maps to combine
list_maps_h5 = []

for file in os.listdir(ICAROS_LOW_BCKG_DIR):

    if any(str(run) in file for run in RUNS_TO_COMBINE) and 'zemrude' in file:

        filename = os.path.join(ICAROS_LOW_BCKG_DIR, file)
        print(f"{os.path.basename(filename)} map will be combined.")

        df = pd.read_hdf(filename, key='krmap/krmap')
        list_maps_h5.append(df)

run_15547.zemrude-test.20250717-26-g211f149.KrDesman.zemrude.h5 map will be combined.
run_15557.zemrude-test.20250717-26-g211f149.KrDesman.zemrude.h5 map will be combined.
run_15546.zemrude-test.20250717-26-g211f149.KrDesman.zemrude.h5 map will be combined.


In [ ]:
# Combine the maps using the weighted average functions
combined_map_df = combine_maps(list_maps_h5)
print("Combination complete.")

# Save the combined map to a HDF5 file
output_filename = f'combined_{RUNS_TO_COMBINE[0]}_{RUNS_TO_COMBINE[-1]}_zemrude_map.h5'
print(f"\nSaving combined map to: {os.path.join(OUTPUT_DIR, output_filename)}")
combined_map_df.to_hdf(os.path.join(OUTPUT_DIR, output_filename), key='krmap', mode='w', format='table')
print("Done.")

Combination complete.

Saving combined map to: /lhome/ific/c/ccortesp/Analysis/NEXT-100/Th_analysis/h5/combined_15546_15557_zemrude_map.h5

Done.


# Energy Scale Evolution

In [8]:
# List of h5 files to use for energy scale time evolution
list_scale_h5 = []

for file in os.listdir(ICAROS_HE_DIR):

    if any(str(run) in file for run in RUNS_FOR_ENERGY_SCALE) and 'zemrude' in file:

        filename = os.path.join(ICAROS_HE_DIR, file)
        print(f"{os.path.basename(filename)} file will be used.")

        df = pd.read_hdf(filename, key='t_evol/t_evol')
        list_scale_h5.append(df)

run_15589.zemrude-test-4-g677d7f98.20250717-26-g211f149.KrDesman.zemrude.h5 file will be used.


In [23]:
# Combine the energy scale dataframes
combined_scale_df = create_full_h5_scale(list_scale_h5)
print("Combination complete.")

# Save the combined file to a HDF5 file
output_filename = f'energy_scale_{RUNS_FOR_ENERGY_SCALE[0]}_{RUNS_FOR_ENERGY_SCALE[-1]}_he.h5'
print(f"\nSaving combined map to: {os.path.join(OUTPUT_DIR, output_filename)}")
combined_scale_df.to_hdf(os.path.join(OUTPUT_DIR, output_filename), key='data', mode='w', format='table')
print("Done.")

Combination complete.

Saving combined map to: /lhome/ific/c/ccortesp/Analysis/NEXT-100/Th_analysis/h5/energy_scale_15589_15589_he.h5
Done.
